<a href="https://colab.research.google.com/github/AksharaBasu/GenAI/blob/main/LLM4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata

keys = ["akshara", "GROQ_API_KEY", "openrouter"]

for name in keys:
    try:
        value = userdata.get(name)
        print(name, "→", "✅ Found" if value else "❌ Empty")
    except Exception as e:
        print(name, "→ ❌ Not accessible")

akshara → ✅ Found
GROQ_API_KEY → ✅ Found
openrouter → ✅ Found


In [ ]:
# ============================================================
# 3-LLM AUTOMATIC FALLBACK SYSTEM
# LangGraph + OpenRouter + Groq + OpenRouter Backup + Gradio
# ============================================================

# ------------------------------------------------------------
# 1. INSTALL LIBRARIES
# ------------------------------------------------------------
!pip install -q -U langgraph langchain-openrouter langchain-groq gradio


# ------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------
import gradio as gr
from typing import TypedDict

from google.colab import userdata

from langchain_openrouter import ChatOpenRouter
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END


# ------------------------------------------------------------
# 3. GET API KEYS FROM COLAB SECRETS
# ------------------------------------------------------------
OPENROUTER_API_KEY = userdata.get("openrouter")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
AKSHARA_API_KEY = userdata.get("akshara")


# ------------------------------------------------------------
# 4. CHECK API KEYS
# ------------------------------------------------------------
print("=" * 60)
print("API KEY STATUS")
print("=" * 60)

print(
    "openrouter      :",
    "✅ Found" if OPENROUTER_API_KEY else "❌ Missing"
)

print(
    "GROQ_API_KEY    :",
    "✅ Found" if GROQ_API_KEY else "❌ Missing"
)

print(
    "akshara         :",
    "✅ Found" if AKSHARA_API_KEY else "❌ Missing"
)

print("=" * 60)


# ------------------------------------------------------------
# 5. CREATE 3 LLMs
# ------------------------------------------------------------

# LLM 1
# Primary model
llm1 = ChatOpenRouter(
    api_key=OPENROUTER_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=2048
)


# LLM 2
# Groq backup
llm2 = ChatGroq(
    api_key=GROQ_API_KEY,
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=2048
)


# LLM 3
# Final backup using the akshara OpenRouter key
llm3 = ChatOpenRouter(
    api_key=AKSHARA_API_KEY,
    model="qwen/qwen3-32b",
    temperature=0.2,
    max_tokens=2048
)


print("✅ 3 LLMs created")


# ============================================================
# 6. STATE
# ============================================================

class State(TypedDict):
    question: str

    answer: str

    selected_llm: str

    error_llm1: str
    error_llm2: str
    error_llm3: str


# ============================================================
# 7. LLM 1 NODE
# ============================================================

def llm1_node(state: State):

    question = state["question"]

    prompt = (
        "You are the primary AI assistant.\n\n"
        "Answer the user's question accurately and clearly.\n"
        "If code is requested, provide working code.\n"
        "If it is an academic question, explain it clearly.\n\n"
        "USER QUESTION:\n"
        + question
    )

    try:

        print("🔵 Trying LLM 1 - OpenRouter...")

        response = llm1.invoke(prompt)

        answer = str(response.content).strip()

        if not answer:
            raise Exception("LLM 1 returned an empty response.")

        print("✅ LLM 1 succeeded")

        return {
            "answer": answer,
            "selected_llm": "LLM 1 - OpenRouter"
        }

    except Exception as e:

        print("❌ LLM 1 failed")

        return {
            "error_llm1": str(e)
        }


# ============================================================
# 8. ROUTE AFTER LLM 1
# ============================================================

def route_after_llm1(state: State):

    if state.get("answer"):
        return "end"

    return "llm2"


# ============================================================
# 9. LLM 2 NODE
# ============================================================

def llm2_node(state: State):

    question = state["question"]

    prompt = (
        "You are the second backup AI assistant.\n\n"
        "The primary AI was unavailable.\n"
        "Answer the user's question yourself.\n"
        "Give a clear and useful answer.\n\n"
        "USER QUESTION:\n"
        + question
    )

    try:

        print("🟠 Trying LLM 2 - Groq...")

        response = llm2.invoke(prompt)

        answer = str(response.content).strip()

        if not answer:
            raise Exception("LLM 2 returned an empty response.")

        print("✅ LLM 2 succeeded")

        return {
            "answer": answer,
            "selected_llm": "LLM 2 - Groq"
        }

    except Exception as e:

        print("❌ LLM 2 failed")

        return {
            "error_llm2": str(e)
        }


# ============================================================
# 10. ROUTE AFTER LLM 2
# ============================================================

def route_after_llm2(state: State):

    if state.get("answer"):
        return "end"

    return "llm3"


# ============================================================
# 11. LLM 3 NODE
# ============================================================

def llm3_node(state: State):

    question = state["question"]

    prompt = (
        "You are the final backup AI assistant.\n\n"
        "The first two AI systems were unavailable.\n"
        "Provide the best possible answer to the user's question.\n"
        "Be accurate, clear and useful.\n\n"
        "USER QUESTION:\n"
        + question
    )

    try:

        print("🟢 Trying LLM 3 - Backup OpenRouter...")

        response = llm3.invoke(prompt)

        answer = str(response.content).strip()

        if not answer:
            raise Exception("LLM 3 returned an empty response.")

        print("✅ LLM 3 succeeded")

        return {
            "answer": answer,
            "selected_llm": "LLM 3 - Backup OpenRouter"
        }

    except Exception as e:

        print("❌ LLM 3 failed")

        return {
            "error_llm3": str(e)
        }


# ============================================================
# 12. BUILD LANGGRAPH
# ============================================================

builder = StateGraph(State)


# Add Nodes
builder.add_node("llm1", llm1_node)
builder.add_node("llm2", llm2_node)
builder.add_node("llm3", llm3_node)


# START → LLM 1
builder.add_edge(
    START,
    "llm1"
)


# LLM 1 → END or LLM 2
builder.add_conditional_edges(
    "llm1",
    route_after_llm1,
    {
        "end": END,
        "llm2": "llm2"
    }
)


# LLM 2 → END or LLM 3
builder.add_conditional_edges(
    "llm2",
    route_after_llm2,
    {
        "end": END,
        "llm3": "llm3"
    }
)


# LLM 3 → END
builder.add_edge(
    "llm3",
    END
)


# Compile graph
graph = builder.compile()

print()
print("=" * 60)
print("✅ LANGGRAPH COMPILED SUCCESSFULLY")
print("=" * 60)


# ============================================================
# 13. MAIN FUNCTION
# ============================================================

def ask_ai(question):

    if not question or not question.strip():

        return (
            "❌ Please enter a question.",
            ""
        )

    question = question.strip()

    print()
    print("=" * 60)
    print("QUESTION")
    print("=" * 60)
    print(question)
    print("=" * 60)

    initial_state = {

        "question": question,

        "answer": "",

        "selected_llm": "",

        "error_llm1": "",
        "error_llm2": "",
        "error_llm3": ""
    }


    try:

        result = graph.invoke(initial_state)


        answer = result.get(
            "answer",
            ""
        )

        selected_llm = result.get(
            "selected_llm",
            ""
        )


        # ----------------------------------------------------
        # SUCCESS
        # ----------------------------------------------------

        if answer:

            print()
            print("=" * 60)
            print("FINAL RESULT")
            print("=" * 60)

            print("Selected:", selected_llm)

            return (
                "✅ " + selected_llm,
                answer
            )


        # ----------------------------------------------------
        # ALL 3 FAILED
        # ----------------------------------------------------

        errors = []


        if result.get("error_llm1"):

            errors.append(
                "LLM 1 Error:\n"
                + result["error_llm1"]
            )


        if result.get("error_llm2"):

            errors.append(
                "LLM 2 Error:\n"
                + result["error_llm2"]
            )


        if result.get("error_llm3"):

            errors.append(
                "LLM 3 Error:\n"
                + result["error_llm3"]
            )


        error_message = (
            "❌ ALL 3 LLMs FAILED\n\n"
            + "\n\n".join(errors)
        )


        return (
            error_message,
            "No LLM was able to answer the question."
        )


    except Exception as e:

        return (
            "❌ Graph Error",
            str(e)
        )


# ============================================================
# 14. GRADIO UI
# ============================================================

with gr.Blocks(
    title="3 LLM Automatic Fallback"
) as demo:

    gr.Markdown(
        """
# 🤖 3-LLM Automatic Fallback System

### LangGraph + Multiple LLMs

**Primary**
➡️ LLM 1 — OpenRouter

⬇️ If LLM 1 fails

**Backup**
➡️ LLM 2 — Groq

⬇️ If LLM 2 fails

**Final Backup**
➡️ LLM 3 — OpenRouter (`akshara`)

---

### 🔀 Automatic Routing

The system automatically uses the **first LLM that successfully responds**.
"""
    )


    question_box = gr.Textbox(
        label="📝 Enter Your Question",
        placeholder="Example: What is LangGraph?",
        lines=4
    )


    ask_button = gr.Button(
        "🚀 Ask AI",
        variant="primary"
    )


    selected_output = gr.Textbox(
        label="🔀 Selected LLM",
        lines=2
    )


    answer_output = gr.Markdown(
        label="💬 Final Answer"
    )


    # Button click
    ask_button.click(
        fn=ask_ai,
        inputs=question_box,
        outputs=[
            selected_output,
            answer_output
        ]
    )


    # Enter key
    question_box.submit(
        fn=ask_ai,
        inputs=question_box,
        outputs=[
            selected_output,
            answer_output
        ]
    )


# ============================================================
# 15. LAUNCH
# ============================================================

print()
print("=" * 60)
print("🚀 STARTING GRADIO")
print("=" * 60)

demo.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 100.6 MB/s eta 0:00:00
API KEY STATUS
openrouter      : ✅ Found
GROQ_API_KEY    : ✅ Found
akshara         : ✅ Found
✅ 3 LLMs created

✅ LANGGRAPH COMPILED SUCCESSFULLY

🚀 STARTING GRADIO
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug


QUESTION
Design a complete smart agriculture system using IoT, machine learning, and computer vision. Explain the system architecture, sensors required, data collection process, ML model selection, crop disease detection workflow, database design, cloud integration, security issues, and provide a Python implementation outline. Also explain how the system can predict crop diseases and recommend irrigation automatically.
🔵 Trying LLM 1 - OpenRouter...
✅ LLM 1 succeeded

FINAL RESULT
Selected: LLM 1 - OpenRouter
